# HIPPIE machine-learning tutorial

Companion notebook to steps 5 and 6 of the HIPPIE Machine Learning page. The page explains each step; this notebook only runs it.

Set `DATA` in the *Load the data* cell to your unzipped split folder.

## Setup

With a GPU, install `torch` first as shown on pytorch.org so you get a CUDA build.

In [ ]:
# pip install pandas numpy scikit-learn biopython torch transformers tqdm ipywidgets matplotlib

## Load the data and check the split

In [ ]:
import pandas as pd

DATA = (
    "hippie_split"  # the unzipped ZIP from step 3; for your own run: "results/hippie"
)
splits = {
    name: pd.read_csv(f"{DATA}/{name}.csv")
    for name in ["train", "val", "test_balanced", "test_realistic"]
}
# columns: protein1, protein2, label (1 = interacts, 0 = negative), score

# sanity check 1: class balance -- about 50 % positives, except test_realistic (about 9 %)
for name, df in splits.items():
    print(f"{name:15s} {len(df):7d} pairs   positives: {df.label.mean():.1%}")


# sanity check 2: held-out proteins that also occur in training -- should be close to 0 %
def proteins(df):
    return set(df.protein1) | set(df.protein2)


train_prots = proteins(splits["train"])
for name in ["val", "test_balanced", "test_realistic"]:
    prots = proteins(splits[name])
    print(
        f"{name:15s} {len(prots & train_prots) / len(prots):.1%} of proteins also in train"
    )

## A — Cheap baseline (scikit-learn)

In [ ]:
import os

import numpy as np
from Bio import SeqIO
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# FASTA headers are UniProt accessions; the ZIP has sequences.fasta at its root
# (if your headers look like sp|P12345|NAME, the split below keeps just the accession)
fasta = f"{DATA}/sequences.fasta"
if not os.path.exists(fasta):  # for your own run
    fasta = f"{DATA}/data/sequences.fasta"
seqs = {
    r.id.split("|")[1] if "|" in r.id else r.id: str(r.seq)
    for r in SeqIO.parse(fasta, "fasta")
}


def usable(df):  # drop pairs whose sequence is missing from the FASTA
    keep = df.protein1.isin(seqs.keys()) & df.protein2.isin(seqs.keys())
    if (~keep).any():
        print(f"dropping {(~keep).sum()} pairs without a sequence")
    return df[keep]


splits = {name: usable(df) for name, df in splits.items()}
train, val = splits["train"], splits["val"]

AA = "ACDEFGHIKLMNPQRSTVWY"


def composition(seq):
    counts = np.array([seq.count(a) for a in AA], dtype=float)
    return np.append(counts / len(seq), np.log(len(seq)))


comp = {p: composition(s) for p, s in seqs.items()}


# symmetric pair features: (A, B) and (B, A) give the same vector
def pair_features(vec, df):
    return np.array(
        [
            np.concatenate([vec[a] + vec[b], np.abs(vec[a] - vec[b])])
            for a, b in df[["protein1", "protein2"]].itertuples(index=False)
        ]
    )


base_clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
base_clf.fit(pair_features(comp, train), train["label"])
print(
    "val AUROC baseline:",
    roc_auc_score(val["label"], base_clf.predict_proba(pair_features(comp, val))[:, 1]),
)

## B — Sequence-based model

In [ ]:
import os
import torch
from sklearn.ensemble import RandomForestClassifier
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

# smallest ESM-2 (8M parameters, 320-d). Bigger and usually better:
# facebook/esm2_t12_35M_UR50D, facebook/esm2_t33_650M_UR50D (GPU recommended)
MODEL = "facebook/esm2_t6_8M_UR50D"
cache = f"embeddings_{MODEL.split('/')[-1]}_{DATA.strip('/').replace('/', '-')}.npz"

if os.path.exists(cache):
    emb = dict(np.load(cache))
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("embedding on", device)
    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModel.from_pretrained(MODEL, add_pooling_layer=False).eval().to(device)

    # sort by length so each batch pads as little as possible
    needed = sorted(
        set().union(*(proteins(df) for df in splits.values())),
        key=lambda p: len(seqs[p]),
    )
    emb, BATCH = {}, 32  # lower BATCH if you run out of (GPU) memory
    for i in tqdm(range(0, len(needed), BATCH), desc="ESM-2", unit="batch"):
        batch = needed[i : i + BATCH]
        chunk = [seqs[p][:1022] for p in batch]  # ESM-2 context limit
        tokens = tokenizer(chunk, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            hidden = model(**tokens).last_hidden_state
        for j, (p, s) in enumerate(zip(batch, chunk)):
            # mean over the residues only: skip the start token and the padding
            emb[p] = hidden[j, 1 : len(s) + 1].mean(0).cpu().numpy()
    np.savez(cache, **emb)


# a random forest prints no progress, so grow it in chunks of trees behind a progress bar
# (same forest as one fit; raise step if you have many more cores than 25)
def fit_forest(X, y, desc, n_trees=300, step=25):
    rf = RandomForestClassifier(
        n_estimators=0, warm_start=True, n_jobs=-1, random_state=0
    )
    for _ in tqdm(
        range(0, n_trees, step), desc=f"{desc} ({n_trees} trees)", unit="step"
    ):
        rf.n_estimators += step
        rf.fit(X, y)
    return rf


X_train, X_val = pair_features(emb, train), pair_features(emb, val)
seq_clf = fit_forest(X_train, train["label"], "sequence model")

# tune hyperparameters on the validation split (never on the test sets)
print(
    "val AUROC sequence:",
    roc_auc_score(val["label"], seq_clf.predict_proba(X_val)[:, 1]),
)

## Control: shuffled labels

In [ ]:
shuffled = train["label"].sample(frac=1, random_state=0).to_numpy()
ctrl = fit_forest(X_train, shuffled, "shuffled control")
print(
    "val AUROC shuffled labels:",
    roc_auc_score(val["label"], ctrl.predict_proba(X_val)[:, 1]),
)

## Step 6: evaluate on both test sets

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score

models = {  # name -> function that scores a table of pairs
    "baseline": lambda df: base_clf.predict_proba(pair_features(comp, df))[:, 1],
    "sequence": lambda df: seq_clf.predict_proba(pair_features(emb, df))[:, 1],
    "shuffled": lambda df: ctrl.predict_proba(pair_features(emb, df))[:, 1],
}
test_sets = ["test_balanced", "test_realistic"]

rows = []
for test_name in test_sets:
    test = splits[test_name]
    y = test["label"].to_numpy()
    for name, score in models.items():
        proba = score(test)
        rows.append(
            {
                "test set": test_name,
                "model": name,
                "AUROC": roc_auc_score(y, proba),
                "AUPRC": average_precision_score(y, proba),
            }
        )
results = pd.DataFrame(rows)
print(results.round(3).to_string(index=False))

# overview plot: one panel per metric, one bar per model
colors = ["#2a78d6", "#eb6834", "#1baf7a"]  # baseline, sequence, shuffled
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, metric in zip(axes, ["AUROC", "AUPRC"]):
    table = results.pivot(index="test set", columns="model", values=metric).loc[
        test_sets, list(models)
    ]
    table.plot.bar(
        ax=ax,
        rot=0,
        width=0.75,
        color=colors,
        edgecolor="white",
        linewidth=2,
        legend=False,
    )
    for x, test_name in enumerate(test_sets):  # random-guess level per test set
        chance = 0.5 if metric == "AUROC" else splits[test_name]["label"].mean()
        ax.hlines(
            chance,
            x - 0.4,
            x + 0.4,
            colors="#555555",
            linestyles="--",
            linewidth=1.5,
            label="random guess" if x == 0 else None,
        )
    ax.set(title=metric, xlabel="", ylim=(0, 1))
    ax.grid(axis="y", alpha=0.3)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("score")
fig.legend(
    *axes[1].get_legend_handles_labels(), loc="upper center", ncol=4, frameon=False
)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig("evaluation.png", dpi=150)
plt.show()